In [1]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

2025-10-17 09:53:34.288237: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760694814.563798      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760694814.636679      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## Load Dataset from Colab Directory

In [2]:
train_path = "/kaggle/input/fer2013/train"
test_path = "/kaggle/input/fer2013/test"

## --- Data Generators ---

In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

test_datagen = ImageDataGenerator(rescale=1./255)
test_data = test_datagen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


## Load Pre-trained ResNet-50 Model

In [14]:
from tensorflow.keras.applications import ResNet50

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))

In [15]:
# Freeze base layers first
for layer in base_model.layers:
    layer.trainable = False

In [16]:
x = Flatten()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(7, activation='softmax')(x)   # 7 classes for facial emotions

In [17]:
model = Model(inputs=base_model.input, outputs=output)


##  Compile The Model

In [18]:
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

## --- Train ---


In [19]:
history = model.fit(train_data, validation_data=test_data, epochs=20)

Epoch 1/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 328s 351ms/step - accuracy: 0.2399 - loss: 1.9976 - val_accuracy: 0.2476 - val_loss: 1.8633
Epoch 2/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 304s 339ms/step - accuracy: 0.2518 - loss: 1.8724 - val_accuracy: 0.2471 - val_loss: 1.8343
Epoch 3/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 298s 332ms/step - accuracy: 0.2534 - loss: 1.8555 - val_accuracy: 0.2471 - val_loss: 1.8091
Epoch 4/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 350s 390ms/step - accuracy: 0.2507 - loss: 1.8483 - val_accuracy: 0.2471 - val_loss: 1.8261
Epoch 5/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 306s 340ms/step - accuracy: 0.2517 - loss: 1.8411 - val_accuracy: 0.2471 - val_loss: 1.8063
Epoch 6/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 301s 335ms/step - accuracy: 0.2535 - loss: 1.8353 - val_accuracy: 0.2471 - val_loss: 1.8088
Epoch 7/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 299s 333ms/step - accuracy: 0.2451 - loss: 1.8365 - val_accuracy: 0.2471 - val_loss: 1.7992
Epoch 8/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 302s 336ms/step - accuracy: 0.2500 -

# Train Another Model (VGG-16) to compare with ResNet-50

In [20]:
from tensorflow.keras.applications import VGG16

# --- Load Pretrained Base Model ---
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Freeze base layers (for feature extraction phase)
for layer in base_model.layers:
    layer.trainable = False



58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## Add Custom Layers on Top


In [21]:
# --- Add Custom Layers on top ---
x = Flatten()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(7, activation='softmax')(x)   # 7 classes (emotions)

model = Model(inputs=base_model.input, outputs=output)



## Compile The Model

In [23]:
# --- Compile the Model ---
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,139,271 (80.64 MB)

 Trainable params: 6,424,583 (24.51 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

# --- Data Augmentation ---

In [24]:

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)



In [25]:
train_data = train_datagen.flow_from_directory(
    train_path,   # 👈 change this path
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    test_path,     # 👈 change this path
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)



Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


# --- Train the Model ---

In [ ]:

history = model.fit(train_data, validation_data=test_data, epochs=20)


Epoch 1/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 331s 357ms/step - accuracy: 0.2885 - loss: 1.7663 - val_accuracy: 0.4379 - val_loss: 1.5330
Epoch 2/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 324s 360ms/step - accuracy: 0.3939 - loss: 1.5679 - val_accuracy: 0.4211 - val_loss: 1.4877
Epoch 3/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 316s 352ms/step - accuracy: 0.4273 - loss: 1.5060 - val_accuracy: 0.4507 - val_loss: 1.4402
Epoch 4/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 312s 347ms/step - accuracy: 0.4359 - loss: 1.4719 - val_accuracy: 0.4638 - val_loss: 1.4180
Epoch 5/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 317s 352ms/step - accuracy: 0.4443 - loss: 1.4402 - val_accuracy: 0.4664 - val_loss: 1.3753
Epoch 6/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 324s 360ms/step - accuracy: 0.4543 - loss: 1.4273 - val_accuracy: 0.4813 - val_loss: 1.3591
Epoch 7/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 327s 364ms/step - accuracy: 0.4659 - loss: 1.4078 - val_accuracy: 0.4696 - val_loss: 1.3687
Epoch 8/20
898/898 ━━━━━━━━━━━━━━━━━━━━ 338s 376ms/step - accuracy: 0.4685 -

# --EfficientNetB0 Transfer Learning code--

In [3]:
from tensorflow.keras.applications import EfficientNetB0

# Load EfficientNetB0 Base Model
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # freeze base layers initially


I0000 00:00:1760677681.079142      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## Data Augmentation & Generators

In [4]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(224, 224),
    batch_size=64,
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    test_path,
    target_size=(224, 224),
    batch_size=64,
    class_mode='categorical'
)


Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


## Add Custom Layers

In [5]:
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(train_data.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)


## Compile The Model

In [6]:
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [7]:
history = model.fit(train_data, validation_data=test_data, epochs=5)

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5


I0000 00:00:1760677751.104917      98 service.cc:148] XLA service 0x7e3ef024ed50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1760677751.105684      98 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1760677752.974880      98 cuda_dnn.cc:529] Loaded cuDNN version 90300


  2/449 ━━━━━━━━━━━━━━━━━━━━ 25s 58ms/step - accuracy: 0.1602 - loss: 2.0118   

I0000 00:00:1760677762.506189      98 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


449/449 ━━━━━━━━━━━━━━━━━━━━ 521s 1s/step - accuracy: 0.2301 - loss: 1.8450 - val_accuracy: 0.2471 - val_loss: 1.8172
Epoch 2/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 331s 738ms/step - accuracy: 0.2404 - loss: 1.8292 - val_accuracy: 0.2471 - val_loss: 1.8143
Epoch 3/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 328s 729ms/step - accuracy: 0.2446 - loss: 1.8237 - val_accuracy: 0.2471 - val_loss: 1.8152
Epoch 4/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 327s 727ms/step - accuracy: 0.2524 - loss: 1.8218 - val_accuracy: 0.2471 - val_loss: 1.8152
Epoch 5/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 328s 731ms/step - accuracy: 0.2472 - loss: 1.8224 - val_accuracy: 0.2471 - val_loss: 1.8146


In [ ]:
history = model.fit(train_data, validation_data=test_data, epochs=5)

Epoch 1/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 327s 727ms/step - accuracy: 0.2509 - loss: 1.8167 - val_accuracy: 0.2471 - val_loss: 1.8140
Epoch 2/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 333s 741ms/step - accuracy: 0.2477 - loss: 1.8196 - val_accuracy: 0.2471 - val_loss: 1.8151
Epoch 3/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 703ms/step - accuracy: 0.2536 - loss: 1.8207

In [ ]:
# history = model.fit(train_data, validation_data=test_data, epochs=5)